In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
print("----- THIRISHA.M-----------")
# Load Dataset
dataset = load_dataset("imdb")

# Preprocess Dataset
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

encoded_dataset = dataset.map(preprocess_function, batched=True)
train_dataset = encoded_dataset["train"].shuffle(seed=42).select(range(10000))
test_dataset = encoded_dataset["test"].shuffle(seed=42).select(range(1000))

In [ ]:
print("-------------------------------")
# Load Pre-trained Model
base_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

In [ ]:
print("---------------------------------------------")
# Configure LoRA
lora_config = LoraConfig(
    task_type="SEQ_CLS",
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

model = prepare_model_for_kbit_training(base_model)
model = get_peft_model(model, lora_config)

# Move the model to GPU if available
model.to(device)

In [ ]:
print("---------------------------------------------")
# Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    logging_dir="./logs",
    logging_steps=10,
    fp16=True,  # Use mixed precision training for faster performance on GPU
)

In [ ]:
print("---------------------------------------------")
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)

In [ ]:
print("---------------------------------------------")
# Train
trainer.train()

In [ ]:
print("---------------------------------------------")
# Save the Fine-Tuned Model
model.save_pretrained("./lora_bert_sentiment")
tokenizer.save_pretrained("./lora_bert_sentiment")